# Notebook 1 — Data Ingestion Test

This notebook verifies that the 3GPP PDF documents load correctly and that
the environment variables (API keys) are accessible. It's a sanity check
before building the full ingestion pipeline in the next notebook.

In [2]:
import sys
print(sys.executable)  # should show rag3gpp in the path

c:\Users\nabee\anaconda3\envs\rag3gpp\python.exe


## Environment variables

Loading API keys from the .env file. We use python-dotenv so the keys
never get hardcoded into the notebook or pushed to GitHub.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file into os.environ

# just checking the keys loaded - never print the actual values
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))

OpenAI key loaded: True
LangSmith key loaded: True


## Loading a 3GPP PDF

Using LangChain's PyPDFLoader to load a single spec file. The loader splits
the document into one Document object per page, each with the page text and
metadata (filename, page number). We print the first 500 characters to confirm
the content looks right.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Projects\3gpp-rag\data\raw\TS_38211.pdf")
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print(f"\nPage 1 metadata: {pages[0].metadata}")
print(f"\nFirst 500 characters of page 1:\n{pages[0].page_content[:500]}")

Total pages loaded: 171

Page 1 metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-13T19:41:29-04:00', 'title': '3GPP TS 38.211', 'author': 'Stefan Parkvall - Ericsson', 'subject': 'NR; Physical channels and modulation (Release 15)', 'keywords': 'NR, Layer 1', 'moddate': '2026-04-13T19:41:29-04:00', 'source': 'C:\\Projects\\3gpp-rag\\data\\raw\\TS_38211.pdf', 'total_pages': 171, 'page': 0, 'page_label': '1'}

First 500 characters of page 1:
3GPP TS 38.211 V19.3.0 (2026-03) 
Technical Specification 
3rd Generation Partnership Project; 
Technical Specification Group Radio Access Network; 
NR; 
Physical channels and modulation 
(Release 19) 
  
 
  
 
The present document has been developed within the 3rd Generation Partnership Project (3GPP TM) and may be further elaborated for the purposes of 3GPP.. 
The present document has not been subject to any approval process by the 3GPP Organizational Partners and s

In [5]:
import os

pdf_files = [
    r"C:\Projects\3gpp-rag\data\raw\TS_38211.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38214.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38300.pdf",
]

for path in pdf_files:
    loader = PyPDFLoader(path)
    pages = loader.load()
    filename = os.path.basename(path)
    print(f"{filename}: {len(pages)} pages, title: {pages[0].metadata.get('title', 'N/A')}")

TS_38211.pdf: 171 pages, title: 3GPP TS 38.211
TS_38214.pdf: 356 pages, title: 3GPP TS 38.214
TS_38300.pdf: 314 pages, title: 3GPP TS 38.300


In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader

raw_dir = r"C:\Projects\3gpp-rag\data\raw"
pdf_files = [f for f in os.listdir(raw_dir) if f.endswith(".pdf")]

total_pages = 0
for filename in sorted(pdf_files):
    path = os.path.join(raw_dir, filename)
    loader = PyPDFLoader(path)
    pages = loader.load()
    total_pages += len(pages)
    print(f"{filename}: {len(pages)} pages")

print(f"\nTotal pages across all specs: {total_pages}")

TS_38211.pdf: 171 pages
TS_38212.pdf: 327 pages
TS_38213.pdf: 331 pages
TS_38214.pdf: 356 pages
TS_38215.pdf: 38 pages
TS_38300.pdf: 314 pages
TS_38331.pdf: 1902 pages
TS_38401.pdf: 184 pages

Total pages across all specs: 3623
